# Sampling flux vacua

> **What's in this notebook?** This notebook collects the workflows for sampling SUSY and non-SUSY flux vacua at scale, built on top of the ISD principle introduced in NB06. We walk through three complementary workflows (canonical SUSY from input fluxes, the wrapper API, the non-SUSY variation), compare the available sampling methods on a single tadpole-bounded benchmark, and end with an advanced performance-tuning section.
>
> **In this notebook, you will learn:**
>
> - How to refine an integer-rounded ISD flux to a true SUSY vacuum via `newton_method_flux_vacua` (Workflow A).
> - How to drive the same pipeline through the high-level `sample_SUSY_flux_vacua` wrapper (Workflow B).
> - How the non-SUSY variation differs in mode flag, sign conventions, and result analysis (Workflow C).
> - How the four ISD-shift methods compare on tadpole regimes and what the bounding-box scaling looks like (Method comparison).
> - How to tune sampling regions, $N_{\max}$ knobs, and hybrid solvers for production runs (Advanced).
>
> **Prerequisites:** [NB04 — Sampling module](../01_basics/04_sampling_module.ipynb), [NB06 — ISD sampling principle](06_ISD_sampling_principle.ipynb), [NB05 — Finding flux vacua](05_finding_flux_vacua.ipynb).

## Outline

1. [Setup](#setup)
2. [Workflow A: SUSY ISD sampling from input fluxes](#workflow-a-susy-isd-sampling-from-input-fluxes)
3. [Workflow B: SUSY ISD sampling via the wrapper API](#workflow-b-susy-isd-sampling-via-the-wrapper-api)
4. [Workflow C: Non-SUSY ISD sampling](#workflow-c-non-susy-isd-sampling)
5. [Method comparison and benchmarks](#method-comparison-and-benchmarks)
6. [Advanced: performance tuning](#advanced-performance-tuning)
7. [Take-aways](#take-aways)
8. [Further reading](#further-reading)

## Setup

In [ ]:
# General imports
import warnings, sys, time
import numpy as np
from tqdm.auto import tqdm
from scipy.optimize import root

# JAX imports
import jax
from jax import jit, vmap
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc
from jaxvacua.util import vmapping_func

warnings.filterwarnings("ignore")

**Canonical fixture.** Throughout this notebook we use the degree-18 hypersurface in $\mathbb{CP}^{1,1,1,6,9}$ ($h^{2,1}=2$, $Q_{D3}=276$, `model_ID=1`) as the running example. The same fixture is reused in every workflow and benchmark so that results are directly comparable. The h12=3 mini-examples at the end of §A and §C illustrate that the workflows generalise without code changes.

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(
    h12=h12,
    Q=276,
    model_ID=1,
    maximum_degree=2,
    limit="LCS",
    model_type="KS",
)

We build a `data_sampler` with mildly restrictive flux and moduli bounds that we then reuse across §A, §B, §C, and the benchmark in §5. The construction is identical to that of NB04 — the sampler is purely infrastructure here.

In [ ]:
sampler = jvc.data_sampler(
    model,
    flux_bounds=[-5, 5],
    moduli_bounds=[0, 10],
    axion_bounds=[-0.5, 0.5],
)

For convenient ensemble evaluation we vectorise the F-term function `model.DW` once and reuse it throughout. The cross-link to the ISD principle: the algorithmic intro to ISD sampling lives in [NB06](06_ISD_sampling_principle.ipynb); here we focus on the *workflow* of producing vacua at scale.

In [ ]:
DW_v = vmap(model.DW)

## Workflow A: SUSY ISD sampling from input fluxes

Workflow A starts from a fixed *integer* flux choice and searches for all moduli configurations that yield a SUSY vacuum compatible with it. This is the complement of Workflow B (moduli-first); together they cover the two natural ways of populating an ensemble. The algorithmic primitive is the same in both — the ISD-rounded shift from NB06 [ISD principle](06_ISD_sampling_principle.ipynb) — but the order in which fluxes and moduli are sampled differs.

### The algorithm

The pipeline has two stages, each `vmap`'d twice (once over moduli starting points and once over candidate flux vectors):

1. `find_solution_init_vmap` — apply `model.linearised_shifts` to round the input flux into an ISD-compatible integer flux and produce near-vacuum starting moduli.
2. `find_solution_steps_vmap` — iterate `model.linearised_shifts` to drive the residual $|D_I W|$ to zero (Newton-like).

Below we wire up both stages, run a few iterations on two flux samples, and confirm the residual shrinks at each step.

In [ ]:
# Common kwargs for the optimiser. "Fflux" fixes the RR-fluxes as input;
# the linearised_shifts call then solves for the NS-NS half.
mode = "Fflux"
kwargs = {"mode": mode, "Q": model.D3_tadpole,
          "constraints": None, "remove_NANs": True}

# Double vmap of model.DW — once over moduli starting points, once over fluxes.
DW_vv = jax.vmap(jax.vmap(model.DW))

The two doubly-vmapped optimisers — one for the ISD initialisation step (`find_solution_init_vmap`) and one for the Newton refinement step (`find_solution_steps_vmap`) — are built with the `vmapping_func` helper imported in §1:

In [ ]:
# Step 1: ISD initialisation. Outer vmap: moduli starting points. Inner: same flux.
find_solution_init = vmapping_func(
    model.linearised_shifts, in_axes=(0, 0, None), return_flag=False, **kwargs)
find_solution_init_vmap = vmapping_func(
    find_solution_init, in_axes=(None, None, 0))

# Step 2: Newton refinement. Both axes vmapped.
find_solution_steps = vmapping_func(
    model.linearised_shifts, in_axes=(0, 0, 0), return_flag=True, **kwargs)
find_solution_steps_vmap = vmapping_func(
    find_solution_steps, in_axes=(0, 0, 0))

We supply two starting points and two candidate flux vectors. The doubly-vmapped call evaluates every (moduli, flux) pair in parallel.

In [ ]:
moduli0 = np.array([
    [ 0.22924045 + 3.00550378j, -0.46016314 + 2.28651085j],
    [-0.04468382 + 0.63834065j,  0.25839392 + 2.36493542j],
])
tau0    = np.array([0.18254317 + 6.81361114j, 0.49828078 + 8.33716844j])
fluxes0 = np.array([[ 1,  5,  1,  2,  4, -1],
                    [ 5, -3,  0, -4, -4, -3]])

# Step 1: ISD initialisation rounds the fluxes and shifts the moduli.
moduli, tau, fluxes = find_solution_init_vmap(moduli0, tau0, fluxes0)

Iterating the Newton-like step `find_solution_steps_vmap` then drives $|D_I W|$ toward zero. We track the per-flux residual at each step:

In [ ]:
moduli1, tau1, fluxes1, _ = find_solution_steps_vmap(moduli, tau, fluxes)

for i in range(25):
    moduli1, tau1, fluxes1, _ = find_solution_steps_vmap(moduli1, tau1, fluxes1)
    DW = DW_vv(moduli1, jnp.conj(moduli1), tau1, jnp.conj(tau1), fluxes1)
    res = np.sum(np.abs(DW), axis=2)
    print(f"step {i:2d}   |DW| flux1: {res[0]}   flux2: {res[1]}")

### Large-scale workflow

All of the above is packaged in a single wrapper, `model.sample_SUSY_vacua_from_fluxes`, which combines the initialisation, the Newton refinement, and the bookkeeping. Below we sample 2,000 starting configurations and refine them in one call (takes a couple of minutes):

In [ ]:
rns_key = jax.random.PRNGKey(42)

moduli, tau, fluxes, residual = model.sample_SUSY_vacua_from_fluxes(
    fluxes_init=fluxes0,
    N=2 * 10**3,
    sampler=sampler,
    rns_key=rns_key,
    max_iters=10,
    objective_fct=DW_vv,
    optimiser_init=find_solution_init_vmap,
    optimiser_steps=find_solution_steps_vmap,
    vmap_dim_pts=10,
    mode=mode,
    print_progress=True,
)

DW = jax.vmap(model.DW)(moduli, jnp.conj(moduli), tau, jnp.conj(tau), fluxes)
print(f"all |DW| residuals below 1e-10: "
      f"{np.all(np.sum(np.abs(DW), axis=1) < 1e-10)}")

### Improved constrained sampling — input fluxes across a moduli region

By passing `fluxes_init=None` we instead let the sampler draw fluxes internally, while still constraining the moduli to a chosen sub-region of the Kähler cone. This is the flux-first analog of moduli-first sampling: instead of fixing fluxes and searching moduli, we fix a region of moduli space and let the sampler explore fluxes consistent with it.

In [ ]:
moduli, tau, fluxes, residuals = model.sample_SUSY_vacua_from_fluxes(
    fluxes_init=None,
    N=100,
    sampler=sampler,
    rns_key=rns_key,
    max_iters=10,
    moduli_sampling_mode="cone",
    max_tadpole=276,
    objective_fct=DW_vv,
    optimiser_init=find_solution_init_vmap,
    optimiser_steps=find_solution_steps_vmap,
    vmap_dim_flux=100,
    vmap_dim_pts=100,
    mode=mode,
    print_progress=True,
)

### Higher-$h^{1,2}$ example: SUSY workflow at $h^{1,2}=3$

The workflow above generalises without code changes to higher $h^{1,2}$. As an illustration we run the same Workflow-A pipeline on a $h^{1,2}=3$ model (`model_ID=4`), reusing the doubly-vmapped optimisers built above — only the model and sampler need refreshing.

In [ ]:
# h12=3 example (model_ID=4 has hyperplanes + sensible sampling region).
model_3 = jvc.FluxVacuaFinder(h12=3, model_ID=4, model_type="KS",
                              maximum_degree=2, limit="LCS")
sampler_3 = jvc.data_sampler(
    model_3,
    flux_bounds=[-5, 5],
    moduli_bounds=(2., 8.),
    dilaton_bounds=(np.sqrt(3) / 2, 10.),
    axion_bounds=(-0.5, 0.5),
    seed=42,
)

# Rebuild the optimisers with the new model.
kwargs_3 = {**kwargs, "Q": model_3.D3_tadpole}
_init_3 = vmapping_func(model_3.linearised_shifts, in_axes=(0, 0, None),
                       return_flag=False, **kwargs_3)
init_3_vmap = vmapping_func(_init_3, in_axes=(None, None, 0))
_step_3 = vmapping_func(model_3.linearised_shifts, in_axes=(0, 0, 0),
                       return_flag=True, **kwargs_3)
step_3_vmap = vmapping_func(_step_3, in_axes=(0, 0, 0))
DW_vv_3 = jax.vmap(jax.vmap(model_3.DW))

moduli3, tau3, fluxes3, res3 = model_3.sample_SUSY_vacua_from_fluxes(
    fluxes_init=None,
    N=50,
    sampler=sampler_3,
    rns_key=jax.random.PRNGKey(42),
    max_iters=10,
    moduli_sampling_mode="cone",
    max_tadpole=276,
    objective_fct=DW_vv_3,
    optimiser_init=init_3_vmap,
    optimiser_steps=step_3_vmap,
    vmap_dim_flux=50,
    vmap_dim_pts=10,
    mode=mode,
    print_progress=True,
)
DW3 = jax.vmap(model_3.DW)(moduli3, jnp.conj(moduli3),
                          tau3, jnp.conj(tau3), fluxes3)
print(f"h12=3 — {len(moduli3)} vacua; "
      f"all |DW| < 1e-10: {np.all(np.sum(np.abs(DW3), axis=1) < 1e-10)}")

## Workflow B: SUSY ISD sampling via the wrapper API

**What's new vs §A.** §A wired the two-stage pipeline manually: build two doubly-vmapped optimisers (`find_solution_init_vmap` and `find_solution_steps_vmap`), call them in sequence, iterate. §B wraps the same pipeline behind a single high-level driver, `model.sample_SUSY_flux_vacua`, that handles the moduli draws, ISD rounding, and Newton refinement in one call. The trade-off: less fine-grained control over the inner loop, but a much shorter user-facing snippet that suffices for routine production runs.

### ISD sampling with a single optimiser

The simplest usage passes one optimiser — here `linearised_shifts_F` (constructed once with `vmapping_func`) — together with the §4 sampler. `vmap_dim` controls the batch size of the inner XLA kernel.

In [ ]:
rns_key = jax.random.PRNGKey(42)

kwargs = {"Q": model.D3_tadpole, "return_flag": True,
          "constraints": None, "remove_NANs": True,
          "in_axes": (0, 0, 0)}

linearised_shifts_F = vmapping_func(
    model.linearised_shifts, mode="Fflux", **kwargs)

# Single-optimiser run targeting N=1000 SUSY vacua.
moduli, tau, fluxes, residuals = model.sample_SUSY_flux_vacua(
    N=10**3,
    rns_key=rns_key,
    sampler=sampler,
    optimiser=linearised_shifts_F,
    objective_fct=DW_v,
    vmap_dim=10**2,
    print_progress=True,
)

### Running with multiple optimisers

Different ISD modes (`"ISD"`, `"Hflux"`, `"Fflux"`) cover different halves of the flux vector. Passing a list as `optimisers=...` lets the wrapper round-robin between them and pool the resulting vacua — handy when one mode alone yields too few convergent points.

In [ ]:
linearised_shifts_ISD = vmapping_func(
    model.linearised_shifts, mode="ISD",   **kwargs)
linearised_shifts_H   = vmapping_func(
    model.linearised_shifts, mode="Hflux", **kwargs)
linearised_shifts_F   = vmapping_func(
    model.linearised_shifts, mode="Fflux", **kwargs)

optimisers = [linearised_shifts_ISD, linearised_shifts_H, linearised_shifts_F]

moduli, tau, fluxes, residuals = model.sample_SUSY_flux_vacua(
    N=10**2,
    rns_key=rns_key,
    sampler=sampler,
    optimisers=optimisers,
    objective_fct=DW_v,
    vmap_dim=10**2,
    print_progress=True,
)

### Constrained sampling — moduli-region restrictions

The wrapper accepts a JIT-compiled boolean `constraints(moduli, tau, flux)` callback that filters vacua at every batch. As a proof of concept we restrict the moduli to $2<\operatorname{Im}(z^i)<3$ and the axio-dilaton to $\operatorname{Im}(\tau)>4$, and rebuild the sampler with matching bounds so the initial guesses sit in the feasible region.

In [ ]:
@jit
def constraints_model(moduli, tau, flux):
    return (jnp.all(jnp.imag(moduli) < 3., axis=0)
            & jnp.all(jnp.imag(moduli) > 2., axis=0)
            & (jnp.imag(tau) > 4))

kwargs_C = {"Q": model.D3_tadpole, "return_flag": True,
            "constraints": constraints_model, "remove_NANs": True,
            "in_axes": (0, 0, 0)}
linearised_shifts_F_v = vmapping_func(
    model.linearised_shifts, mode="Fflux", **kwargs_C)

# Tighter sampler matching the constraint region.
sampler_constrained = jvc.data_sampler(
    model, flux_bounds=[-5, 5],
    moduli_bounds=[2, 3], dilaton_bounds=[4, 10])

In [ ]:
moduli, tau, fluxes, residuals = model.sample_SUSY_flux_vacua(
    N=10**2,
    rns_key=rns_key,
    sampler=sampler_constrained,
    optimiser=linearised_shifts_F_v,
    objective_fct=DW_v,
    vmap_dim=10**2,
    print_progress=True,
)

## Workflow C: Non-SUSY ISD sampling

*(Section content to be populated in S8d — sourced from `19_non_susy_sampling.ipynb`. The h12=3 mini-example follows at the end.)*

## Method comparison and benchmarks

*(Section content to be populated in S8e — sourced from `18_sampling_comparison.ipynb` plus the 'Old method: uniform sampling' carry-over from NB8.)*

## Advanced: performance tuning

*(Section content to be populated in S8e (or a dedicated sub-step) — sourced from NB19 §10, the seven-subsection performance-analysis block.)*

## Take-aways

*(Final bullets written after the workflow content lands so they reflect what actually ships.)*

## Further reading

- [NB04 — Sampling module](../01_basics/04_sampling_module.ipynb)
- [NB05 — Finding flux vacua](05_finding_flux_vacua.ipynb)
- [NB06 — ISD sampling principle](06_ISD_sampling_principle.ipynb)
- [NB10 — Flux bounding](../03_flux_bounding/10_flux_bounding.ipynb) (constrained enumeration within a tadpole bound)